Notebook goal:

- Reindex cluster IDs to remove missing IDs in the ID sequence originating from removing clusters posthoc.
- Map new IDs into marker genes csv.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path 
import tempfile
import os

import numpy as np
import pandas as pd
import scanpy as sc
from dotenv import dotenv_values

from project_utils import FileTracker

In [3]:
CONFIG = dotenv_values("config.env") 

# Dirs
DATADIR = Path(CONFIG['DATADIR'])
CLUSTERING_DIR = DATADIR / "5_clustering"
MODELDIR = CLUSTERING_DIR / 'all_control_posthoc_qc'
DATASETS_DIR = MODELDIR / "datasets"

# file paths
ADATA_FP = DATASETS_DIR / 'raw.h5ad'
EMBEDDING_FP = DATASETS_DIR / 'embed_only.h5ad'

ADATA_OUT_FP = DATASETS_DIR / "control_clustered.h5ad"
LABEL_OUT_FP = CLUSTERING_DIR / "final_labelmat_qc.csv"
MAPPER_OUT_FP = CLUSTERING_DIR / "old_to_reindexed_mapper.csv"

MARKERS_FP = CLUSTERING_DIR / 'HypoMap3D_marker_genes_qc.csv'

tracker = FileTracker(log_file=DATADIR/'tracked_files.toml')

### Let's load the anndata

In [4]:
adata = sc.read_h5ad(Path(MODELDIR, 'datasets', 'raw.h5ad'))
embed_only = sc.read_h5ad(Path(MODELDIR, 'datasets', 'embed_only.h5ad'), backed='r')

adata.obsm = embed_only[adata.obs_names].obsm
adata.uns = {k: v for k, v in embed_only.uns.items() if k in {'tsne', 'umap', 'neighbors'}}
adata.obsp = embed_only.obsp

del embed_only
adata

AnnData object with n_obs × n_vars = 898050 × 56939
    obs: 'CellID', 'Dataset', 'Sample', 'total_counts', 'n_genes', 'pct_mito', 'Author', 'Source', 'Data_source_link', 'GEO', 'Publication_Link', 'Suspension', 'Technology', 'Author_Sex_harmonized', 'Strain', 'Diet', 'Treatment', 'Age', 'Mean_age_in_weeks', 'IsControl', 'cluster_id_label', 'supertype_id_label', 'subclass_id_label', 'class_id_label', 'C66_named', 'C185_named', 'C286_named', 'C465_named', 'ABC_class_propagated', 'ABC_class_confidence', 'to_be_removed', 'IsNeuronal', 'major_features', 'major_features_propagated', 'major_features_confidence', 'ABC_supertype_propagated', 'ABC_supertype_confidence', 'ABC_subclass_propagated', 'ABC_subclass_confidence', 'ABC_cluster_propagated', 'ABC_cluster_confidence', 'L1', 'L2', 'L3', 'L4', 'L5'
    var: 'gene_ids', 'IsMouseTF', 'IsHumanTFOrtholog', 'IsSourceGene', 'IsTargetGene', 'IsHvgExcludedGene', 'IsAnnotationExcludedGene', 'IsPriorityGene', 'IsTopGene', 'IsHvg'
    uns: 'neighbors'

## Let's get stable IDs



PLAN:

1. calculate cluster 1d umap to order the clusters according to their similarity.

2. Run the following for neurons and non-neurons separately:

- for L1:
    give id based on centroid coordinate derived from averaging the umap coordinates of all cells of a cluster.

- for L2:
    start with L1-1, get children
    give id based on centroid coordinate derived from averaging the umap coordinates of all cells of a cluster.
    repeat for each parent cluster.

etc etc


### 1D umap

In [ ]:

sc.pp.neighbors(
    adata,
    n_neighbors=15,
    use_rep="X_embed"
)
    
sc.tl.umap(adata, key_added='X_umap_1d', n_components=1)


### neurons

In [12]:
neurons = adata[~adata.obs['L1'].str.startswith('NN')].copy()
neurons

AnnData object with n_obs × n_vars = 607093 × 56939
    obs: 'CellID', 'Dataset', 'Sample', 'total_counts', 'n_genes', 'pct_mito', 'Author', 'Source', 'Data_source_link', 'GEO', 'Publication_Link', 'Suspension', 'Technology', 'Author_Sex_harmonized', 'Strain', 'Diet', 'Treatment', 'Age', 'Mean_age_in_weeks', 'IsControl', 'cluster_id_label', 'supertype_id_label', 'subclass_id_label', 'class_id_label', 'C66_named', 'C185_named', 'C286_named', 'C465_named', 'ABC_class_propagated', 'ABC_class_confidence', 'to_be_removed', 'IsNeuronal', 'major_features', 'major_features_propagated', 'major_features_confidence', 'ABC_supertype_propagated', 'ABC_supertype_confidence', 'ABC_subclass_propagated', 'ABC_subclass_confidence', 'ABC_cluster_propagated', 'ABC_cluster_confidence', 'L1', 'L2', 'L3', 'L4', 'L5'
    var: 'gene_ids', 'IsMouseTF', 'IsHumanTFOrtholog', 'IsSourceGene', 'IsTargetGene', 'IsHvgExcludedGene', 'IsAnnotationExcludedGene', 'IsPriorityGene', 'IsTopGene', 'IsHvg'
    uns: 'neighbors'

In [13]:
# Extract labelmat as numpy arr
keys = [f"L{i}" for i in range(1,6)]
labelmat = np.asarray(neurons.obs[keys])

In [14]:
# Compute centroids per level, not across all levels
centroid_labelmat = np.zeros(labelmat.shape, dtype=np.float32)

for j in range(labelmat.shape[1]):  # loop over levels / columns
    col = labelmat[:, j]

    # unique labels only within this level
    unique_labels, inverse = np.unique(col, return_inverse=True)

    # centroid for each label in this level
    centroids = np.zeros(len(unique_labels), dtype=np.float32)
    for i, label in enumerate(unique_labels):
        mask = (col == label)
        centroids[i] = np.mean(neurons.obsm['X_umap_1d'][mask])

    # map back into this column
    centroid_labelmat[:, j] = centroids[inverse]

In [15]:
# safely get unique centroid paths
centroid_pathmat, path_indices = np.unique(centroid_labelmat, return_index=True, axis=0)
pathmat = labelmat[path_indices]
# safer, in case some centroids for different clusters are identical 
path_centroids = centroid_labelmat[path_indices]


In [16]:
# get number of paths and number of levels 
n_paths, n_levels = pathmat.shape
#generate 0 matrix
stable_path_ids = np.zeros((n_paths, n_levels), dtype=int)

# L1 stable IDs
# get current l1 cluster names and the centroids 
l1_raw = pathmat[:, 0]
l1_cent = path_centroids[:, 0]

#for each unique l1 cluster get the centroid value and store in an list
unique_l1 = np.unique(l1_raw)
l1_info = []
for raw in unique_l1:
    idx = np.where(l1_raw == raw)[0][0]
    l1_info.append((raw, l1_cent[idx]))


In [18]:

# sort in order of centroids 
l1_info.sort(key=lambda x: x[1])
# assign ID to each raw cluster 
l1_map = {raw: i+1 for i, (raw, _) in enumerate(l1_info)}
#add to stable id paths 
stable_path_ids[:, 0] = np.array([l1_map[x] for x in l1_raw])

In [20]:
#deeper levels 
for level in range(1, n_levels):
    #get parent stable ids
    parent_ids = stable_path_ids[:, level-1]
    # get children raw cluster names 
    child_raw = pathmat[:, level]
    # get children centroids 
    child_cent = path_centroids[:, level]
    
    #iterate through unique paths
    seen = {}
    for i in range(n_paths):
        # get parent stable id and child raw id 
        key = (parent_ids[i], child_raw[i])
        #get unique paths for only these levels
        if key not in seen:
            seen[key] = child_cent[i]

    # sort by parent stable ID then child raw id 
    ordered = sorted(seen.items(), key=lambda x: (x[0][0], x[1]))

    # order children now by cluster centroid
    child_map = {
        key: new_id
        for new_id, (key, _) in enumerate(ordered, start=1)
    }

    # generate child stable ids 
    stable_path_ids[:, level] = np.array([
        child_map[(parent_ids[i], child_raw[i])]
        for i in range(n_paths)
    ])

# map back to all observations
stable_labelmat = np.zeros_like(labelmat, dtype=int)
stable_labelmat[:, 0] = np.array([l1_map[x] for x in labelmat[:, 0]])

for level in range(1, n_levels):
    mapping = {}
    for i in range(n_paths):
        key = (stable_path_ids[i, level - 1], pathmat[i, level])
        mapping[key] = stable_path_ids[i, level]

    stable_labelmat[:, level] = np.array([
        mapping[(stable_labelmat[i, level - 1], labelmat[i, level])]
        for i in range(labelmat.shape[0])
    ])


In [21]:
stable_labelmat

array([[  1,  35, 113, 290, 390],
       [  1,  33, 107, 274, 371],
       [  1,  35, 111, 286, 386],
       ...,
       [  1,  12,  29,  73,  95],
       [  1,  33, 106, 268, 365],
       [  1,  15,  45, 114, 150]], shape=(607093, 5))

In [30]:
# save back
for j, key in enumerate(keys):
    neurons.obs[f'L{j+1}_ID'] = stable_labelmat[:, j]
    neurons.obs[f'L{j+1}_ID'] = neurons.obs[f'L{j+1}_ID'].astype(str)

In [ ]:
# prepend Gb/Gl and level idx
gb_mask = neurons.obs.L1.str.contains("GABA")
gl_mask = neurons.obs.L1.str.contains("Glut")

for i in range(1,6):
    neurons.obs.loc[gb_mask, f"L{i}_ID"] = neurons.obs.loc[gb_mask, f"L{i}_ID"].apply(lambda x: "Gb" + str(i) + "-" + str(x))
    neurons.obs.loc[gl_mask, f"L{i}_ID"] = neurons.obs.loc[gl_mask, f"L{i}_ID"].apply(lambda x: "Gl" + str(i) + "-" + str(x))

In [60]:
neurons.obs.loc[:, "L1_ID"] = neurons.obs.apply(lambda row: row["L1_ID"] + " " + row["L1"].split(" ")[1], axis=1)

### Non-neurons

In [34]:
glia = adata[adata.obs['L1'].str.startswith('NN')].copy()
glia

AnnData object with n_obs × n_vars = 290957 × 56939
    obs: 'CellID', 'Dataset', 'Sample', 'total_counts', 'n_genes', 'pct_mito', 'Author', 'Source', 'Data_source_link', 'GEO', 'Publication_Link', 'Suspension', 'Technology', 'Author_Sex_harmonized', 'Strain', 'Diet', 'Treatment', 'Age', 'Mean_age_in_weeks', 'IsControl', 'cluster_id_label', 'supertype_id_label', 'subclass_id_label', 'class_id_label', 'C66_named', 'C185_named', 'C286_named', 'C465_named', 'ABC_class_propagated', 'ABC_class_confidence', 'to_be_removed', 'IsNeuronal', 'major_features', 'major_features_propagated', 'major_features_confidence', 'ABC_supertype_propagated', 'ABC_supertype_confidence', 'ABC_subclass_propagated', 'ABC_subclass_confidence', 'ABC_cluster_propagated', 'ABC_cluster_confidence', 'L1', 'L2', 'L3', 'L4', 'L5'
    var: 'gene_ids', 'IsMouseTF', 'IsHumanTFOrtholog', 'IsSourceGene', 'IsTargetGene', 'IsHvgExcludedGene', 'IsAnnotationExcludedGene', 'IsPriorityGene', 'IsTopGene', 'IsHvg'
    uns: 'neighbors'

In [ ]:
# Extract labelmat as numpy arr
labelmat = np.asarray(glia.obs[keys])
# Compute centroids per level, not across all levels
centroid_labelmat = np.zeros(labelmat.shape, dtype=np.float32)

for j in range(labelmat.shape[1]):  # loop over levels / columns
    col = labelmat[:, j]

    # unique labels only within this level
    unique_labels, inverse = np.unique(col, return_inverse=True)

    # centroid for each label in this level
    centroids = np.zeros(len(unique_labels), dtype=np.float32)
    for i, label in enumerate(unique_labels):
        mask = (col == label)
        centroids[i] = np.mean(glia.obsm['X_umap_1d'][mask])

    # map back into this column
    centroid_labelmat[:, j] = centroids[inverse]

# safely get unique centroid paths
centroid_pathmat, path_indices = np.unique(centroid_labelmat, return_index=True, axis=0)
pathmat = labelmat[path_indices]
# safer, in case some centroids for different clusters are identical 
path_centroids = centroid_labelmat[path_indices]

# get number of paths and nmber of levels (592 clusters, 5 levels)
n_paths, n_levels = pathmat.shape
#generate 0 matrix
stable_path_ids = np.zeros((n_paths, n_levels), dtype=int)
# L1 stable IDs
# get current l1 cluster names and the centroids 
l1_raw = pathmat[:, 0]
l1_cent = path_centroids[:, 0]
#for each unique l1 cluster get the centroid value and store in an list
unique_l1 = np.unique(l1_raw)
l1_info = []
for raw in unique_l1:
    idx = np.where(l1_raw == raw)[0][0]
    l1_info.append((raw, l1_cent[idx]))
# sort in order of centroids 
l1_info.sort(key=lambda x: x[1])
# assign ID to each raw cluster 
l1_map = {raw: i+1 for i, (raw, _) in enumerate(l1_info)}
#add to stable id paths 
stable_path_ids[:, 0] = np.array([l1_map[x] for x in l1_raw])

#deeper levels 
for level in range(1, n_levels):
    #get parent stable ids
    parent_ids = stable_path_ids[:, level-1]
    # get children raw cluster names 
    child_raw = pathmat[:, level]
    # get children centroids 
    child_cent = path_centroids[:, level]
    
    #iterate through unique paths
    seen = {}
    for i in range(n_paths):
        # get parent stable id and child raw id 
        key = (parent_ids[i], child_raw[i])
        #get unique paths for only these levels
        if key not in seen:
            seen[key] = child_cent[i]

    # sort by parent stable ID then child raw id 
    ordered = sorted(seen.items(), key=lambda x: (x[0][0], x[1]))

    # order children now by cluster centroid
    child_map = {
        key: new_id
        for new_id, (key, _) in enumerate(ordered, start=1)
    }

    # generate child stable ids 
    stable_path_ids[:, level] = np.array([
        child_map[(parent_ids[i], child_raw[i])]
        for i in range(n_paths)
    ])

# map back to all observations
stable_labelmat = np.zeros_like(labelmat, dtype=int)
stable_labelmat[:, 0] = np.array([l1_map[x] for x in labelmat[:, 0]])

for level in range(1, n_levels):
    mapping = {}
    for i in range(n_paths):
        key = (stable_path_ids[i, level - 1], pathmat[i, level])
        mapping[key] = stable_path_ids[i, level]

    stable_labelmat[:, level] = np.array([
        mapping[(stable_labelmat[i, level - 1], labelmat[i, level])]
        for i in range(labelmat.shape[0])
    ])

# save back
for j, key in enumerate(keys):
    glia.obs[f'L{j+1}_ID'] = stable_labelmat[:, j]
    glia.obs[f'L{j+1}_ID'] = glia.obs[f'L{j+1}_ID'].astype(str)

for i in range(1,6):
    glia.obs.loc[:, f"L{i}_ID"] = glia.obs.loc[:, f"L{i}_ID"].apply(lambda x: "NN" + str(i) + "-" + str(x))

In [56]:
glia.obs.loc[:, "L1_ID"] = glia.obs.apply(lambda row: row["L1_ID"] + " " + row["L1"].split(" ")[1], axis=1)

In [57]:
glia.obs.L1_ID.unique()

array(['NN1-2 Immune', 'NN1-3 Vascular', 'NN1-1 Oligo',
       'NN1-4 Astro-Epen'], dtype=object)

### Let's create a label_df and add it back to the initial adata

In [61]:
id_cols = [f"L{i}_ID" for i in range(1,6)]

label_df = pd.concat([
    neurons.obs.loc[:, id_cols],
    glia.obs.loc[:, id_cols]
],axis=0)

label_df = label_df.loc[adata.obs.index, :]
label_df.columns = [f"L{i}" for i in range(1,6)]

In [62]:
label_df

,L1,L2,L3,L4,L5
SC1-01_SAMN18754353_AAACCCACAACCAGAG,Gl1-1 Glut,Gl2-35,Gl3-113,Gl4-290,Gl5-390
SC1-01_SAMN18754353_AAACCCAGTCATCTAG,Gl1-1 Glut,Gl2-33,Gl3-107,Gl4-274,Gl5-371
SC1-01_SAMN18754353_AAACGAAAGCAAGCCA,Gl1-1 Glut,Gl2-35,Gl3-111,Gl4-286,Gl5-386
SC1-01_SAMN18754353_AAACGAAGTGCGGCTT,Gl1-1 Glut,Gl2-35,Gl3-113,Gl4-291,Gl5-391
SC1-01_SAMN18754353_AAACGAATCGCATAGT,Gl1-1 Glut,Gl2-35,Gl3-114,Gl4-297,Gl5-398
...,...,...,...,...,...
SC2-25_SAMN46505609_TTTGTCTAGCAAGACA,NN1-4 Astro-Epen,NN2-12,NN3-22,NN4-34,NN5-34
SC2-25_SAMN46505609_TTTGTGAAGCTTACTT,NN1-4 Astro-Epen,NN2-12,NN3-22,NN4-34,NN5-34
SC2-25_SAMN46505609_TTTGTGGCAATAGTCT,NN1-4 Astro-Epen,NN2-12,NN3-22,NN4-34,NN5-34
SC2-25_SAMN46505609_TTTGTGGCACAAAGCG,NN1-4 Astro-Epen,NN2-13,NN3-25,NN4-38,NN5-38


In [64]:
label_df.L1.unique()

array(['Gl1-1 Glut', 'Gb1-2 GABA', 'NN1-2 Immune', 'NN1-3 Vascular',
       'NN1-1 Oligo', 'NN1-4 Astro-Epen'], dtype=object)

## Let's get the IDs

### Let's create a mapper from old to reindexed 

In [50]:
def build_label_mapper(label_df, labelmat_df, levels=["L1", "L2", "L3", "L4", "L5"]):
    """
    Builds a flat mapper from old labels to new labels at each hierarchical level.
    
    Parameters
    ----------
    label_df : pd.DataFrame
        DataFrame with new cluster labels, indexed by cell names, columns L1-L5.
    labelmat_df : pd.DataFrame
        DataFrame with old cluster labels, indexed by cell names, columns L1-L5.
    levels : list
        List of column names representing the hierarchy levels.
    
    Returns
    -------
    mapper : dict
        Flat dict of the form {(level, old_label): new_label}.
    """
    
    common_cells = label_df.index.intersection(labelmat_df.index)
    print(f"Number of common cells: {len(common_cells)}")
    print(f"Cells only in label_df: {len(label_df.index.difference(labelmat_df.index))}")
    print(f"Cells only in labelmat_df: {len(labelmat_df.index.difference(label_df.index))}")

    new_labels = label_df.loc[common_cells]
    old_labels = labelmat_df.loc[common_cells]

    mapper = {}

    for level in levels:
        if level not in new_labels.columns or level not in old_labels.columns:
            print(f"Level {level} not found in both dataframes, skipping.")
            continue

        for old_label in old_labels[level].unique():
            cells_with_old_label = old_labels[level] == old_label
            new_labels_for_cells = new_labels.loc[cells_with_old_label, level]
            unique_new_labels = new_labels_for_cells.unique()
            
            if len(unique_new_labels) > 1:
                value_counts = new_labels_for_cells.value_counts()
                most_common_new_label = value_counts.idxmax()
                print(f"WARNING: {level} | old label '{old_label}' maps to multiple new labels: {value_counts.to_dict()}. Using most common: '{most_common_new_label}'.")
                mapper[old_label] = most_common_new_label
            else:
                mapper[old_label] = unique_new_labels[0]

    return mapper

In [65]:
mapper = build_label_mapper(label_df, adata.obs.loc[:, [f"L{i}" for i in range(1,6)]])
mapper_df = pd.DataFrame(
    mapper.items(),
    columns=["old_label", "reindexed_label"]
)

Number of common cells: 898050
Cells only in label_df: 0
Cells only in labelmat_df: 0


In [ ]:
########## DEV: add back in a legacy reindex version 
try:
    old_mapper_df = pd.read_csv(MAPPER_OUT_FP)
    old_mapper = old_mapper_df.set_index("old_label").to_dict()["reindexed_label"]
    mapper_df["legacy_reindexed_label"] = mapper_df.old_label.map(old_mapper)
except:
    print("No old mapper_df available.")

In [69]:
mapper_df

,old_label,reindexed_label,legacy_reindexed_label
0,Gl1-1 Glut,Gl1-1 Glut,Gl1-2 Glut
1,Gb1-2 GABA,Gb1-2 GABA,Gb1-1 GABA
2,NN1-5 Immune,NN1-2 Immune,NN1-4 Immune
3,NN1-4 Vascular,NN1-3 Vascular,NN1-5 Vascular
4,NN1-6 Oligo,NN1-1 Oligo,NN1-3 Oligo
...,...,...,...
2005,NN5-0983,NN5-40,NN5-2004
2006,NN5-0988,NN5-35,NN5-1999
2007,NN5-0348,NN5-31,NN5-1997
2008,NN5-0987,NN5-36,NN5-2000


### Let's save

In [70]:
# labels
label_df.to_csv(LABEL_OUT_FP)
# mapper
mapper_df.to_csv(MAPPER_OUT_FP, index=False) 

In [71]:
# add new IDs to adata
adata.obs.loc[:, [f"L{i}" for i in range(1,6)]] = label_df.values
del adata.obsm['X_umap_1d'], adata.uns["X_umap_1d"]

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [ ]:
with tracker.register(ADATA_OUT_FP) as tf:
    tf.set_description('Control Hypomap3D with low quality clusters/cells removed and reindexed cluster IDs.')
    adata.write_h5ad(tf.path)

### Let's map into the new markers

In [73]:
# map new IDs into markers 
mapper_df = pd.read_csv(MAPPER_OUT_FP)

#mapper = mapper_df.set_index("old_label").to_dict()["reindexed_label"] # use this line if no legacy reindexing had been performed for the markers yet
mapper = mapper_df.set_index("legacy_reindexed_label").to_dict()["reindexed_label"]

# Load existing file
markers = pd.read_csv(MARKERS_FP)

markers.cluster = markers.cluster.map(mapper)

assert markers.cluster.isna().sum() == 0

In [74]:

# Atomically replace cell history to avoid data corruption
output_path = MARKERS_FP
output_dir = output_path.parent
# Create temp file in same directory
with tempfile.NamedTemporaryFile(mode="w", delete=False, dir=output_dir, suffix=".csv") as tmp:
    temp_path = Path(tmp.name)
    markers.to_csv(tmp.name, index=False)

os.replace(temp_path, output_path)